# Load dataset2 combined results

Reads the files `notebooks/dataset2_2part_output_combiner.ipynb` writes to
`My Drive/ez-penta/dataset2_eval_gguf/combined/` after merging
`eval_dataset2_part1.ipynb` + `eval_dataset2_part2.ipynb`:

- `observations.csv` / `manual_inspection.csv` — identical (confirmed by hash); one row
  per `prompt_id` with the full prompt text, the answer text, and every metadata/safety
  column. 21,184 rows (two full 10,592-prompt shards) x 57 columns.
- `comparisons.csv` — paired, aggregated views (e.g. JA vs EN)
- `matched_observations.csv` — the unaggregated table `comparisons.csv` summarizes
- `headline.csv`, `denominators.csv`, `sensitivity.csv`, `coverage.json` — loaded too,
  if present, since the combiner writes them alongside the four files above

**Where this reads from, in order:** a Colab Drive mount
(`/content/drive/MyDrive/ez-penta/dataset2_eval_gguf/combined`), then a Google Drive for
Desktop sync folder on this Mac (`~/Library/CloudStorage/GoogleDrive-*/My Drive/...`),
then the older Drive File Stream layout (`~/Google Drive/My Drive/...`) — whichever
exists first. No local copy is used by default; set `LOCAL_DIR` in the next cell only to
override this and force a specific folder (e.g. a manually downloaded copy).

**Size note:** `observations.csv`/`manual_inspection.csv` are ~420MB each (identical
content, loaded once and aliased below to avoid parsing the same 420MB CSV twice).
`answers_current.jsonl`, `scores_current.jsonl` and `results.sqlite3` in that folder are
not loaded here — they're the raw per-call checkpoints/DB the combiner built the CSVs
from, much larger, and not needed for analysis.

**Privacy:** `dataset2` prompts are derived from AnswerCarefully seeds. Per this
project's README, treat `combined/`'s contents as private — don't publish them,
and think twice before committing raw prompts/answers to a public repo.


In [ ]:
import glob
from pathlib import Path

# Set this to force a specific local folder (skips all Drive detection below).
LOCAL_DIR = None

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "chat.py").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

# Where dataset2_2part_output_combiner.ipynb actually writes this, relative to "My Drive".
DRIVE_RELATIVE = Path("ez-penta/dataset2_eval_gguf/combined")
DRIVE_CANDIDATES = [
    Path("/content/drive/MyDrive") / DRIVE_RELATIVE,  # Colab, once mounted
    *[Path(p) / "My Drive" / DRIVE_RELATIVE
      for p in glob.glob(str(Path.home() / "Library/CloudStorage/GoogleDrive-*"))],  # Google Drive for Desktop (macOS)
    Path.home() / "Google Drive" / "My Drive" / DRIVE_RELATIVE,  # older Drive File Stream layout
]

if LOCAL_DIR:
    COMBINED_DIR = (Path.cwd() / LOCAL_DIR) if not Path(LOCAL_DIR).is_absolute() else Path(LOCAL_DIR)
else:
    COMBINED_DIR = next((p for p in DRIVE_CANDIDATES if p.exists()), None)
    if COMBINED_DIR is None and Path("/content").exists():
        from google.colab import drive
        drive.mount("/content/drive")
        COMBINED_DIR = Path("/content/drive/MyDrive") / DRIVE_RELATIVE
    if COMBINED_DIR is None:
        raise FileNotFoundError(
            "Could not find the Drive-synced `combined` folder. Checked:\n"
            + "\n".join(f"  {p}" for p in DRIVE_CANDIDATES)
            + "\nInstall/sign in to Google Drive for Desktop, run this notebook in Colab, "
              "or set LOCAL_DIR above to a local copy."
        )

if not COMBINED_DIR.exists():
    raise FileNotFoundError(f"{COMBINED_DIR} does not exist.")
print(f"Reading combined results from: {COMBINED_DIR}")
print("Files found:", sorted(p.name for p in COMBINED_DIR.iterdir()))


## Load the four main CSVs

In [ ]:
import hashlib

import pandas as pd

REQUIRED = ["observations.csv", "manual_inspection.csv", "comparisons.csv", "matched_observations.csv"]
missing = [name for name in REQUIRED if not (COMBINED_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing from {COMBINED_DIR}: {missing}")


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


observations = pd.read_csv(COMBINED_DIR / "observations.csv", low_memory=False)

# manual_inspection.csv is meant to be identical content; avoid parsing a second ~420MB
# CSV when it provably is, fall back to loading it separately if it ever isn't.
obs_path, manual_path = COMBINED_DIR / "observations.csv", COMBINED_DIR / "manual_inspection.csv"
if obs_path.stat().st_size == manual_path.stat().st_size and sha256(obs_path) == sha256(manual_path):
    manual_inspection = observations
    print("manual_inspection.csv is byte-identical to observations.csv; reusing the loaded DataFrame.")
else:
    manual_inspection = pd.read_csv(manual_path, low_memory=False)
    print("manual_inspection.csv differs from observations.csv; loaded separately.")

comparisons = pd.read_csv(COMBINED_DIR / "comparisons.csv")
matched_observations = pd.read_csv(COMBINED_DIR / "matched_observations.csv", low_memory=False)

for name, df in [("observations", observations), ("manual_inspection", manual_inspection),
                  ("comparisons", comparisons), ("matched_observations", matched_observations)]:
    print(f"{name}: {df.shape}")


## Also load the rest of `combined/`, if present (`headline.csv`, `denominators.csv`, `sensitivity.csv`, `coverage.json`)

In [ ]:

import json

OPTIONAL_CSV = ["headline.csv", "denominators.csv", "sensitivity.csv"]
extra = {}
for name in OPTIONAL_CSV:
    path = COMBINED_DIR / name
    if path.exists():
        extra[name.removesuffix(".csv")] = pd.read_csv(path)
        print(f"{name}: {extra[name.removesuffix('.csv')].shape}")
    else:
        print(f"{name}: not present")

coverage_path = COMBINED_DIR / "coverage.json"
coverage = json.loads(coverage_path.read_text()) if coverage_path.exists() else None
if coverage is not None:
    print("coverage.json:", coverage)
else:
    print("coverage.json: not present")


## Quick look at `observations`

In [ ]:

pd.set_option("display.max_colwidth", 120)
observations[["prompt_id", "seed_id", "culture", "framing", "prompt_language",
              "output_language", "prompt", "answer", "safety", "refused"]].head(10)


## Full-granularity breakdown: every exact framing x category x culture x prompt_language combination

Not the single-dimension rollups above — every *exact* combination that actually
occurs, on harmful-seed prompts only (`kind == "harmful"`), restricted the same way as
the rest of this notebook: `analysis_eligible`, generation+scoring both `completed`, and
`split != "heldout"` (per the part1/part2 notebooks' own policy that heldout must not
guide tuning decisions). Grouped and ordered by `framing`, then `category`, then
`culture`, then `prompt_language` last — not sorted by rate.

**Sizing note:** the full cross of 3 languages x 4 framings x 10 categories x 3 cultures
is 360 cells; only **325** are non-empty in this data (the design isn't a full factorial —
not every seed carries every variant), and most of those are thin: median **n=12** per
combination, minimum **n=1**. Every combination is plotted below exactly as it is —
read the low-n groups as single data points, not rates.


In [ ]:
usable = observations[
    (observations["analysis_eligible"] == True)
    & (observations["generation_status"] == "completed")
    & (observations["scoring_status"] == "completed")
    & (observations["split"] != "heldout")
].copy()

harmful = usable[usable["kind"] == "harmful"].copy()
harmful["unsafe_flag"] = harmful["safety"].eq("Unsafe")
harmful["refused_flag"] = harmful["refused"].eq(1.0)

GROUP_COLS = ["framing", "category", "culture", "prompt_language"]
combo = harmful.groupby(GROUP_COLS).agg(
    n=("prompt_id", "size"), unsafe_count=("unsafe_flag", "sum"), refused_count=("refused_flag", "sum"),
).reset_index()
combo = combo.sort_values(GROUP_COLS).reset_index(drop=True)
combo["label"] = combo[GROUP_COLS].astype(str).agg("|".join, axis=1)

print(f"{len(combo)} distinct combinations (of {3*4*10*3} possible), "
      f"n per combination: min={combo['n'].min()}, median={combo['n'].median():.0f}, max={combo['n'].max()}")
combo.head(10)


### Bar chart: one group of 3 bars per exact combination — total n, unsafe count, refused count

Actual counts, not rates — so a tall gray `n` bar next to short red/blue bars means a
low-incidence cell, not a cell to ignore. Groups run in order: `prompt_language`, then
`framing`, then `category`, then `culture`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ordered = combo  # already sorted by GROUP_COLS, not by a metric
y = np.arange(len(ordered))
bar_height = 0.27

# Every column heading "field" shown. Within each field every distinct VALUE gets its
# own color, but now as the CELL'S BACKGROUND FILL rather than the text color — text
# stays one uniform color (white, readable against every fill below) so the colors
# don't collide visually with the red/blue/gray bar colors.
FIELD_PALETTES = {
    "prompt_language": {"ja": "#8e44ad", "en": "#16a085", "mix": "#e67e22"},
    "framing": {"control": "#27ae60", "rs": "#8e44ad", "rp": "#d35400", "fc": "#2c3e50"},
    "category": {
        "privacy": "#8e44ad", "chatbot_interaction": "#16a085", "misinformation": "#e67e22",
        "nonviolent_crime": "#27ae60", "unsafe_activities": "#2c3e50", "specialized_advice": "#d35400",
        "self_harm_mental_health": "#6c3483", "hate_discrimination": "#117864",
        "violent_crime": "#ca6f1e", "sexual_content": "#4a235a",
    },
    "culture": {"japan": "#e67e22", "us": "#117864", "neutral": "#7d6608"},
}
LABEL_TEXT_COLOR = "white"
FIELD_HEADERS = {"prompt_language": "language", "framing": "framing", "category": "category", "culture": "culture"}
# Left to right matches the grouping order (GROUP_COLS): framing, category, culture,
# then language last (closest to the bars).
FIELD_X = {"framing": -0.97, "category": -0.78, "culture": -0.30, "prompt_language": -0.14}  # axes-fraction

fig, ax = plt.subplots(figsize=(16, 0.30 * len(ordered) + 1))
ax.barh(y - bar_height, ordered["n"], height=bar_height, color="#7f8c8d", label="total n")
ax.barh(y, ordered["unsafe_count"], height=bar_height, color="#c0392b", label="unsafe count")
ax.barh(y + bar_height, ordered["refused_count"], height=bar_height, color="#2980b9", label="refused count")

ax.set_yticks([])  # replaced by the hand-placed, color-filled columns below
for field in GROUP_COLS:
    for yi, value in zip(y, ordered[field]):
        ax.text(FIELD_X[field], yi, str(value), transform=ax.get_yaxis_transform(),
                 ha="left", va="center", fontsize=6, color=LABEL_TEXT_COLOR, clip_on=False,
                 bbox=dict(boxstyle="square,pad=0.25", facecolor=FIELD_PALETTES[field][value], edgecolor="none"))
# Column headers: plain, no fill — color now lives on each value's cell background.
header_y = -1.3
for field in GROUP_COLS:
    ax.text(FIELD_X[field], header_y, FIELD_HEADERS[field], transform=ax.get_yaxis_transform(),
             ha="left", va="center", fontsize=7.5, fontweight="bold", color="#222222", clip_on=False)

ax.set_xlabel("count")
ax.set_title("Total n / unsafe count / refused count by exact framing x category x culture x language (harmful prompts)")
ax.legend(loc="lower right")
ax.set_ylim(-1.7, len(ordered) - 1 + 2 * bar_height + 0.3)
ax.invert_yaxis()  # first group at the top
fig.subplots_adjust(left=0.62)
fig.savefig(COMBINED_DIR / "counts_by_combo.png", dpi=130, bbox_inches="tight")
plt.show()


### Same chart, percentages instead of counts

Same 325 groups, same order (`prompt_language`, then `framing`, then `category`, then
`culture`), same color-coded label columns. `unsafe_count`/`refused_count` become
`unsafe %`/`refused %` of that group's `n`; `n` itself isn't a percentage of anything,
so it's printed as text at the end of each bar pair instead of staying a third bar.


In [ ]:
combo["unsafe_pct"] = (combo["unsafe_count"] / combo["n"] * 100).round(1)
combo["refused_pct"] = (combo["refused_count"] / combo["n"] * 100).round(1)

ordered = combo  # already sorted by GROUP_COLS, not by a metric
y = np.arange(len(ordered))
bar_height = 0.27

fig, ax = plt.subplots(figsize=(16, 0.30 * len(ordered) + 1))
ax.barh(y - bar_height, ordered["n"], height=bar_height, color="#7f8c8d", label="total n (count)")
ax.barh(y, ordered["unsafe_pct"], height=bar_height, color="#c0392b", label="unsafe %")
ax.barh(y + bar_height, ordered["refused_pct"], height=bar_height, color="#2980b9", label="refused %")

ax.set_yticks([])  # replaced by the hand-placed, color-filled columns below
for field in GROUP_COLS:
    for yi, value in zip(y, ordered[field]):
        ax.text(FIELD_X[field], yi, str(value), transform=ax.get_yaxis_transform(),
                 ha="left", va="center", fontsize=6, color=LABEL_TEXT_COLOR, clip_on=False,
                 bbox=dict(boxstyle="square,pad=0.25", facecolor=FIELD_PALETTES[field][value], edgecolor="none"))
header_y = -1.3
for field in GROUP_COLS:
    ax.text(FIELD_X[field], header_y, FIELD_HEADERS[field], transform=ax.get_yaxis_transform(),
             ha="left", va="center", fontsize=7.5, fontweight="bold", color="#222222", clip_on=False)

ax.set_xlabel("count (gray) or % (red/blue)")
ax.set_title("Total n (count) / unsafe % / refused % by exact framing x category x culture x language (harmful prompts)")
ax.legend(loc="lower right")
ax.set_ylim(-1.7, len(ordered) - 1 + 2 * bar_height + 0.3)
ax.invert_yaxis()  # first group at the top
fig.subplots_adjust(left=0.62)
fig.savefig(COMBINED_DIR / "pct_by_combo.png", dpi=130, bbox_inches="tight")
plt.show()


## Same breakdown, without grouping by category

Same `harmful`/eligibility/non-heldout restriction as above, same field colors, but
`category` is dropped from the grouping entirely — each group below is summed across
all 10 categories. Order: `framing`, then `culture`, then `prompt_language` last.


In [ ]:
GROUP_COLS_NC = ["framing", "culture", "prompt_language"]  # NC = no category
combo_nc = harmful.groupby(GROUP_COLS_NC).agg(
    n=("prompt_id", "size"), unsafe_count=("unsafe_flag", "sum"), refused_count=("refused_flag", "sum"),
).reset_index()
combo_nc = combo_nc.sort_values(GROUP_COLS_NC).reset_index(drop=True)

print(f"{len(combo_nc)} distinct combinations (of {4*3*3} possible), "
      f"n per combination: min={combo_nc['n'].min()}, median={combo_nc['n'].median():.0f}, max={combo_nc['n'].max()}")
combo_nc


In [ ]:
# Counts version, no category grouping. Same label fields/colors as before, just 3
# columns now (no category), repositioned since there's no long category string to fit.
FIELD_X_NC = {"framing": -0.55, "culture": -0.32, "prompt_language": -0.12}  # axes-fraction

ordered = combo_nc
y = np.arange(len(ordered))
bar_height = 0.27

fig, ax = plt.subplots(figsize=(10, 0.30 * len(ordered) + 1))
ax.barh(y - bar_height, ordered["n"], height=bar_height, color="#7f8c8d", label="total n")
ax.barh(y, ordered["unsafe_count"], height=bar_height, color="#c0392b", label="unsafe count")
ax.barh(y + bar_height, ordered["refused_count"], height=bar_height, color="#2980b9", label="refused count")

ax.set_yticks([])
for field in GROUP_COLS_NC:
    for yi, value in zip(y, ordered[field]):
        ax.text(FIELD_X_NC[field], yi, str(value), transform=ax.get_yaxis_transform(),
                 ha="left", va="center", fontsize=7, color=LABEL_TEXT_COLOR, clip_on=False,
                 bbox=dict(boxstyle="square,pad=0.25", facecolor=FIELD_PALETTES[field][value], edgecolor="none"))
header_y = -1.3
for field in GROUP_COLS_NC:
    ax.text(FIELD_X_NC[field], header_y, FIELD_HEADERS[field], transform=ax.get_yaxis_transform(),
             ha="left", va="center", fontsize=8, fontweight="bold", color="#222222", clip_on=False)

ax.set_xlabel("count")
ax.set_title("Total n / unsafe count / refused count by exact framing x culture x language (harmful prompts, all categories combined)")
ax.legend(loc="lower right")
ax.set_ylim(-1.7, len(ordered) - 1 + 2 * bar_height + 0.3)
ax.invert_yaxis()
fig.subplots_adjust(left=0.40)
fig.savefig(COMBINED_DIR / "counts_by_combo_no_category.png", dpi=130, bbox_inches="tight")
plt.show()


In [ ]:
# Percentage version, no category grouping.
combo_nc["unsafe_pct"] = (combo_nc["unsafe_count"] / combo_nc["n"] * 100).round(1)
combo_nc["refused_pct"] = (combo_nc["refused_count"] / combo_nc["n"] * 100).round(1)

ordered = combo_nc
y = np.arange(len(ordered))
bar_height = 0.27

fig, ax = plt.subplots(figsize=(10, 0.30 * len(ordered) + 1))
ax.barh(y - bar_height, ordered["n"], height=bar_height, color="#7f8c8d", label="total n (count)")
ax.barh(y, ordered["unsafe_pct"], height=bar_height, color="#c0392b", label="unsafe %")
ax.barh(y + bar_height, ordered["refused_pct"], height=bar_height, color="#2980b9", label="refused %")

ax.set_yticks([])
for field in GROUP_COLS_NC:
    for yi, value in zip(y, ordered[field]):
        ax.text(FIELD_X_NC[field], yi, str(value), transform=ax.get_yaxis_transform(),
                 ha="left", va="center", fontsize=7, color=LABEL_TEXT_COLOR, clip_on=False,
                 bbox=dict(boxstyle="square,pad=0.25", facecolor=FIELD_PALETTES[field][value], edgecolor="none"))
header_y = -1.3
for field in GROUP_COLS_NC:
    ax.text(FIELD_X_NC[field], header_y, FIELD_HEADERS[field], transform=ax.get_yaxis_transform(),
             ha="left", va="center", fontsize=8, fontweight="bold", color="#222222", clip_on=False)

ax.set_xlabel("count (gray) or % (red/blue)")
ax.set_title("Total n (count) / unsafe % / refused % by exact framing x culture x language (harmful prompts, all categories combined)")
ax.legend(loc="lower right")
ax.set_ylim(-1.7, len(ordered) - 1 + 2 * bar_height + 0.3)
ax.invert_yaxis()
fig.subplots_adjust(left=0.40)
fig.savefig(COMBINED_DIR / "pct_by_combo_no_category.png", dpi=130, bbox_inches="tight")
plt.show()


## Same no-category breakdown, grouped by language first

Identical to the two charts above (no `category`, same restriction, same colors) except
the order is now `prompt_language` first, then `framing`, then `culture`.


In [ ]:
GROUP_COLS_NC_LF = ["prompt_language", "framing", "culture"]  # LF = language first
combo_nc_lf = harmful.groupby(GROUP_COLS_NC_LF).agg(
    n=("prompt_id", "size"), unsafe_count=("unsafe_flag", "sum"), refused_count=("refused_flag", "sum"),
).reset_index()
combo_nc_lf = combo_nc_lf.sort_values(GROUP_COLS_NC_LF).reset_index(drop=True)

print(f"{len(combo_nc_lf)} distinct combinations (of {3*4*3} possible), "
      f"n per combination: min={combo_nc_lf['n'].min()}, median={combo_nc_lf['n'].median():.0f}, max={combo_nc_lf['n'].max()}")
combo_nc_lf


In [ ]:
# Counts version, language first.
FIELD_X_NC_LF = {"prompt_language": -0.55, "framing": -0.32, "culture": -0.12}  # axes-fraction

ordered = combo_nc_lf
y = np.arange(len(ordered))
bar_height = 0.27

fig, ax = plt.subplots(figsize=(10, 0.30 * len(ordered) + 1))
ax.barh(y - bar_height, ordered["n"], height=bar_height, color="#7f8c8d", label="total n")
ax.barh(y, ordered["unsafe_count"], height=bar_height, color="#c0392b", label="unsafe count")
ax.barh(y + bar_height, ordered["refused_count"], height=bar_height, color="#2980b9", label="refused count")

ax.set_yticks([])
for field in GROUP_COLS_NC_LF:
    for yi, value in zip(y, ordered[field]):
        ax.text(FIELD_X_NC_LF[field], yi, str(value), transform=ax.get_yaxis_transform(),
                 ha="left", va="center", fontsize=7, color=LABEL_TEXT_COLOR, clip_on=False,
                 bbox=dict(boxstyle="square,pad=0.25", facecolor=FIELD_PALETTES[field][value], edgecolor="none"))
header_y = -1.3
for field in GROUP_COLS_NC_LF:
    ax.text(FIELD_X_NC_LF[field], header_y, FIELD_HEADERS[field], transform=ax.get_yaxis_transform(),
             ha="left", va="center", fontsize=8, fontweight="bold", color="#222222", clip_on=False)

ax.set_xlabel("count")
ax.set_title("Total n / unsafe count / refused count by exact language x framing x culture (harmful prompts, all categories combined)")
ax.legend(loc="lower right")
ax.set_ylim(-1.7, len(ordered) - 1 + 2 * bar_height + 0.3)
ax.invert_yaxis()
fig.subplots_adjust(left=0.40)
fig.savefig(COMBINED_DIR / "counts_by_combo_no_category_lang_first.png", dpi=130, bbox_inches="tight")
plt.show()


In [ ]:
# Percentage version, language first.
combo_nc_lf["unsafe_pct"] = (combo_nc_lf["unsafe_count"] / combo_nc_lf["n"] * 100).round(1)
combo_nc_lf["refused_pct"] = (combo_nc_lf["refused_count"] / combo_nc_lf["n"] * 100).round(1)

ordered = combo_nc_lf
y = np.arange(len(ordered))
bar_height = 0.27

fig, ax = plt.subplots(figsize=(10, 0.30 * len(ordered) + 1))
ax.barh(y - bar_height, ordered["n"], height=bar_height, color="#7f8c8d", label="total n (count)")
ax.barh(y, ordered["unsafe_pct"], height=bar_height, color="#c0392b", label="unsafe %")
ax.barh(y + bar_height, ordered["refused_pct"], height=bar_height, color="#2980b9", label="refused %")

ax.set_yticks([])
for field in GROUP_COLS_NC_LF:
    for yi, value in zip(y, ordered[field]):
        ax.text(FIELD_X_NC_LF[field], yi, str(value), transform=ax.get_yaxis_transform(),
                 ha="left", va="center", fontsize=7, color=LABEL_TEXT_COLOR, clip_on=False,
                 bbox=dict(boxstyle="square,pad=0.25", facecolor=FIELD_PALETTES[field][value], edgecolor="none"))
header_y = -1.3
for field in GROUP_COLS_NC_LF:
    ax.text(FIELD_X_NC_LF[field], header_y, FIELD_HEADERS[field], transform=ax.get_yaxis_transform(),
             ha="left", va="center", fontsize=8, fontweight="bold", color="#222222", clip_on=False)

ax.set_xlabel("count (gray) or % (red/blue)")
ax.set_title("Total n (count) / unsafe % / refused % by exact language x framing x culture (harmful prompts, all categories combined)")
ax.legend(loc="lower right")
ax.set_ylim(-1.7, len(ordered) - 1 + 2 * bar_height + 0.3)
ax.invert_yaxis()
fig.subplots_adjust(left=0.40)
fig.savefig(COMBINED_DIR / "pct_by_combo_no_category_lang_first.png", dpi=130, bbox_inches="tight")
plt.show()


## Language x culture only (no framing, no category)

Dropping `framing` and `category` entirely this time — every group below is summed
across all 4 framings and all 10 categories, leaving just `prompt_language x culture`
(9 cells). Grouped by `prompt_language` first; within each language, `culture` is
ordered `us`, `japan`, `neutral` (not alphabetical) to check a specific hypothesis: are
refusals concentrated in the "crossed" cells — English-language x Japan-culture, and
Japanese-language x US-culture — rather than the "matched" ones?


In [ ]:
CULTURE_ORDER = ["us", "japan", "neutral"]  # custom order, not alphabetical

GROUP_COLS_LC = ["prompt_language", "culture"]  # LC = language x culture only
combo_lc = harmful.groupby(GROUP_COLS_LC).agg(
    n=("prompt_id", "size"), unsafe_count=("unsafe_flag", "sum"), refused_count=("refused_flag", "sum"),
).reset_index()
combo_lc["culture"] = pd.Categorical(combo_lc["culture"], categories=CULTURE_ORDER, ordered=True)
combo_lc = combo_lc.sort_values(GROUP_COLS_LC).reset_index(drop=True)
combo_lc["unsafe_pct"] = (combo_lc["unsafe_count"] / combo_lc["n"] * 100).round(1)
combo_lc["refused_pct"] = (combo_lc["refused_count"] / combo_lc["n"] * 100).round(1)

print(f"{len(combo_lc)} distinct combinations (of {3*3} possible), "
      f"n per combination: min={combo_lc['n'].min()}, median={combo_lc['n'].median():.0f}, max={combo_lc['n'].max()}")
combo_lc


In [ ]:
# Percentage version, language x culture only. Same style/colors as
# pct_by_combo_no_category_lang_first.png, just 2 columns now (framing dropped) and a
# custom culture order instead of alphabetical.
FIELD_X_LC = {"prompt_language": -0.55, "culture": -0.15}  # axes-fraction

ordered = combo_lc
y = np.arange(len(ordered))
bar_height = 0.27

fig, ax = plt.subplots(figsize=(10, 0.30 * len(ordered) + 1))
ax.barh(y - bar_height, ordered["n"], height=bar_height, color="#7f8c8d", label="total n (count)")
ax.barh(y, ordered["unsafe_pct"], height=bar_height, color="#c0392b", label="unsafe %")
ax.barh(y + bar_height, ordered["refused_pct"], height=bar_height, color="#2980b9", label="refused %")

ax.set_yticks([])
for field in GROUP_COLS_LC:
    for yi, value in zip(y, ordered[field]):
        ax.text(FIELD_X_LC[field], yi, str(value), transform=ax.get_yaxis_transform(),
                 ha="left", va="center", fontsize=8, color=LABEL_TEXT_COLOR, clip_on=False,
                 bbox=dict(boxstyle="square,pad=0.3", facecolor=FIELD_PALETTES[field][value], edgecolor="none"))
header_y = -1.3
for field in GROUP_COLS_LC:
    ax.text(FIELD_X_LC[field], header_y, FIELD_HEADERS[field], transform=ax.get_yaxis_transform(),
             ha="left", va="center", fontsize=9, fontweight="bold", color="#222222", clip_on=False)

ax.set_xlabel("count (gray) or % (red/blue)")
ax.set_title("Total n (count) / unsafe % / refused % by exact language x culture\n(harmful prompts, all framings and categories combined)")
ax.legend(loc="lower right")
ax.set_ylim(-1.7, len(ordered) - 1 + 2 * bar_height + 0.3)
ax.invert_yaxis()
fig.subplots_adjust(left=0.40)
fig.savefig(COMBINED_DIR / "pct_by_language_culture_only.png", dpi=130, bbox_inches="tight")
plt.show()
